# `mammos-ai` quickstart

`mammos-ai` contains a collection of pre-trained ai models.

In [1]:
import mammos_ai
import mammos_entity as me

## Simple surrogate micromagnetic model
We can use `mammos-ai` to predict extrinsic magnetic properties of a permanent magnet based on its micromagnetic parameters. Here we use a model to predict coercivity, remanence, and maximum energy product based on saturation magnetization, exchange stiffness, and uniaxial anisotropy constant.

By default this uses the version 1.0 random forest model trained on a 50 nm cubic grain with anisotropy axis aligned with the external field and also aligned with one of the cube axes. Currently only this model is available, but more models will be added in future releases.

Model files are downloaded from the [Hugging Face model repository](https://huggingface.co/mammos-project/mammos-ai-models). More information about the training data can be found in the [training repository](https://github.com/MaMMoS-project/ML-models/tree/main/beyond-stoner-wohlfarth/single-grain-easy-axis-model) and the associated metadata.

Note: Be carful when using models outside of their training data range (this information is available in the model metadata). Predictions outside of the training data range will be unreliable.

Model files are automatically downloaded from the Hugging Face Hub. During the first download (or when checking for updates), you may see the following warning:

> Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

This warning is expected and can be safely ignored.

In [2]:
Ms = me.Entity("SpontaneousMagnetization", 1e6)
A = me.Entity("ExchangeStiffnessConstant", 1e-12)
K1 = me.Entity("MagnetocrystallineAnisotropyConstantK1", 1e6)

extrinsic = mammos_ai.Hc_Mr_BHmax_from_Ms_A_K1(Ms, A, K1)
extrinsic

ExtrinsicProperties(Hc=Entity(ontology_label='CoercivityHcExternal', value=np.float32(970424.6), unit='A / m'), Mr=Entity(ontology_label='Remanence', value=np.float32(999445.25), unit='A / m'), BHmax=Entity(ontology_label='MaximumEnergyProduct', value=np.float32(313311.34), unit='J / m3'))

We can access the individual extrinsic properties as follows:

In [3]:
extrinsic.Hc

Entity(ontology_label='CoercivityHcExternal', value=np.float32(970424.6), unit='A / m')

In [4]:
extrinsic.Mr

Entity(ontology_label='Remanence', value=np.float32(999445.25), unit='A / m')

In [5]:
extrinsic.BHmax

Entity(ontology_label='MaximumEnergyProduct', value=np.float32(313311.34), unit='J / m3')

Internally, the this function checks if the sample is a hard magnet based on the intrinsic properties. If it is, it uses a hard-magnet predictor. If it is not, it switches to a soft-magnet predictor.

`mammos-ai` also provides a function in order to predict if the micromagnetic parameters correspond to a hard magnetic material

In [6]:
mammos_ai.is_hard_magnet_from_Ms_A_K1(Ms, A, K1)

array(True, dtype=object)

We can see if we reduce the anisotropy constant that the material is no longer classified as a hard magnet.

In [7]:
K1 = me.Entity("MagnetocrystallineAnisotropyConstantK1", 1e4)

mammos_ai.is_hard_magnet_from_Ms_A_K1(Ms, A, K1)

array(False, dtype=object)

The the models have metadata associated with them that can be accessed as follows:

In [8]:
mammos_ai.Hc_Mr_BHmax_from_Ms_A_K1_metadata()

{'model_name': 'cube50_singlegrain_random_forest_v1.0',
 'description': 'Random forest model trained on extended simulated data for single grain cubic particles with 50 nm edge length with the external field applied parallel to the anisotropy axis.',
 'training_data_range': {'Ms': (Entity(ontology_label='SpontaneousMagnetization', value=np.float64(79577.47150262764), unit='A / m'),
   Entity(ontology_label='SpontaneousMagnetization', value=np.float64(3978873.5751313814), unit='A / m')),
  'A': (Entity(ontology_label='ExchangeStiffnessConstant', value=np.float64(1e-13), unit='J / m'),
   Entity(ontology_label='ExchangeStiffnessConstant', value=np.float64(1e-11), unit='J / m')),
  'K1': (Entity(ontology_label='MagnetocrystallineAnisotropyConstantK1', value=np.float64(10000.0), unit='J / m3'),
   Entity(ontology_label='MagnetocrystallineAnisotropyConstantK1', value=np.float64(10000000.0), unit='J / m3'))},
 'input_parameters': ['Ms (A/m)', 'A (J/m)', 'K1 (J/m^3)'],
 'output_parameters': [

In [9]:
mammos_ai.is_hard_magnet_from_Ms_A_K1_metadata()

{'model_name': 'cube50_singlegrain_random_forest_v1.0',
 'description': 'Random forest model trained on extended simulated data for single grain cubic particles with 50 nm edge length with the external field applied parallel to the anisotropy axis.',
 'training_data_range': {'Ms': (Entity(ontology_label='SpontaneousMagnetization', value=np.float64(79577.47150262764), unit='A / m'),
   Entity(ontology_label='SpontaneousMagnetization', value=np.float64(3978873.5751313814), unit='A / m')),
  'A': (Entity(ontology_label='ExchangeStiffnessConstant', value=np.float64(1e-13), unit='J / m'),
   Entity(ontology_label='ExchangeStiffnessConstant', value=np.float64(1e-11), unit='J / m')),
  'K1': (Entity(ontology_label='MagnetocrystallineAnisotropyConstantK1', value=np.float64(10000.0), unit='J / m3'),
   Entity(ontology_label='MagnetocrystallineAnisotropyConstantK1', value=np.float64(10000000.0), unit='J / m3'))},
 'input_parameters': ['Ms (A/m)', 'A (J/m)', 'K1 (J/m^3)'],
 'output_classes': {0: 

## Using other models

The package comes with different types of models focused on different tasks. When different models solve the same task, the user can specify which model they want to use.

For example, for single grain cubic hard magnets we have the above mentioned random forest model and a symbolic regression model (further information about the model in the [symbolic regression notebook](./symbolic_regression.ipynb)). The user can simulate extrinsic properties from the intrinsic properties by specifying the `model` argument in the function call, for example running:

In [10]:
mammos_ai.Hc_Mr_BHmax_from_Ms_A_K(Ms, A, K, model="cube50_singlegrain_random_forest_v1.0")

ExtrinsicProperties(Hc=Entity(ontology_label='CoercivityHcExternal', value=np.float32(11173.047), unit='A / m'), Mr=Entity(ontology_label='Remanence', value=np.float32(48390.38), unit='A / m'), BHmax=Entity(ontology_label='MaximumEnergyProduct', value=np.float32(84911.34), unit='J / m3'))

or

In [11]:
mammos_ai.Hc_Mr_BHmax_from_Ms_A_K(Ms, A, K, model="cube50_singlegrain_symbolic_regression_v1.0")

ExtrinsicProperties(Hc=Entity(ontology_label='CoercivityHcExternal', value=np.float64(-45471.12715143046), unit='A / m'), Mr=Entity(ontology_label='Remanence', value=np.float64(-31419732.484784205), unit='A / m'), BHmax=Entity(ontology_label='MaximumEnergyProduct', value=np.float64(720206676.8045163), unit='J / m3'))